# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

import sys
import tempfile
import urllib.request

PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

# Load the dataset metadata
dataset = load_dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata,'cite_as', getattr(metadata,'citeAs', None))}")

## 2. Data Overview
The dataset has one record set, the export of a label-free quantitative proteomics analysis (Proteome Discoverer) of the extracellular vesicles released by human mast cells, unstimulated (**U**) or stimulated (**T**), with three replicates each (samples F1–F6). For each protein, it gives its identification (accession, description, sequence coverage, number of peptides and PSMs, molecular weight, isoelectric point) and its normalized abundance in each sample.

In [ ]:
import urllib.parse
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

record_set = metadata.record_sets[0]
display(pd.DataFrame([{'Field': urllib.parse.unquote_plus(field.name), 'Description': field.description}
                      for field in record_set.fields])
        .style.hide(axis='index').set_caption(f'Fields of {urllib.parse.unquote_plus(record_set.name)}'))

## 3. Data Extraction
The export is nested: each protein row is followed by a header row and the rows of its peptides, which use the same columns for other values (confidence, annotated sequence, modifications, quantification usage...). We load the record set (field names as columns, text decoded, as `mlcroissant` names the columns by field `@id` and returns bytes), then separate the two levels: the **proteins** (the rows with an accession) and their **peptides**, named after the peptide header row. Every field is declared as text in the metadata: the numbers are converted.

In [ ]:
table = pd.DataFrame(dataset.records(record_set=record_set.id))
table = table.rename(columns={field.id: urllib.parse.unquote_plus(field.name) for field in record_set.fields})
table = table.apply(lambda column: column.map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v))
table = table.replace({'': np.nan, 'nan': np.nan})

SAMPLES = [column for column in table.columns if column.startswith('Abundances (Normalized)')]
SAMPLE_NAMES = {column: column.split(': ')[1] + ' ' + column.split(', ')[1] + column.split(', ')[2] for column in SAMPLES}  # e.g. 'F1 U1'
UNSTIMULATED = [c for c in SAMPLES if ', U, ' in c]
STIMULATED = [c for c in SAMPLES if ', T, ' in c]

is_protein = table['Accession'].notna()
is_header = table['Description'].eq('Confidence')

# Proteins
proteins = table[is_protein].copy()
NUMBERS = ['Coverage [%]', '# Peptides', '# PSMs', '# Unique Peptides', 'MW [kDa]', 'calc. pI'] + SAMPLES
proteins[NUMBERS] = proteins[NUMBERS].apply(pd.to_numeric, errors='coerce')
proteins['Gene'] = proteins['Description'].str.extract(r'GN=(\S+)', expand=False)
proteins = proteins.set_index('Accession')

# Peptides, named after the peptide header row, with the accession of their protein
peptide_columns = [name if isinstance(name, str) else 'Protein row' for name in table[is_header].iloc[0]]
peptides = table[~is_protein & ~is_header].copy()
peptides.columns = peptide_columns
peptides['Protein'] = table['Accession'].ffill()[~is_protein & ~is_header]

display(pd.DataFrame({'Value': {
    'Rows of the record set': f'{len(table):,}',
    'Proteins': f'{len(proteins):,}',
    'Peptide rows': f'{len(peptides):,}',
    'Peptide rows used for quantification': f"{peptides['Quan Usage'].eq('Used').mean():.0%}",
}}).style.set_properties(**{'text-align': 'right'}).set_caption('Loaded data'))

## 4. Exploratory Data Analysis (EDA)
### 4.1 Protein identification

In [ ]:
quantified = proteins[SAMPLES].notna()
display(pd.DataFrame({'Value': {
    'Proteins identified': f'{len(proteins):,}',
    'With a gene name': f"{proteins['Gene'].notna().mean():.1%}",
    'Quantified in at least one sample': f'{quantified.any(axis=1).sum():,}',
    'Quantified in all six samples': f'{quantified.all(axis=1).sum():,}',
}}).style.set_properties(**{'text-align': 'right'}).set_caption('Proteins'))

identification = proteins[['Coverage [%]', '# Peptides', '# Unique Peptides', '# PSMs', 'MW [kDa]', 'calc. pI']]
display(identification.describe(percentiles=[0.25, 0.5, 0.75]).T[['min', '25%', '50%', '75%', 'max']]
        .rename(columns={'50%': 'median'}).style.format('{:,.1f}').set_caption('Identification metrics of the proteins'))

### 4.2 Normalizing: log₂ abundances and replicate agreement
Protein abundances span several orders of magnitude: they are compared on a log₂ scale. The correlation between the samples measures how well the replicates agree.

In [ ]:
log2 = np.log2(proteins[SAMPLES]).rename(columns=SAMPLE_NAMES)
per_sample = pd.DataFrame({'Proteins quantified': proteins[SAMPLES].notna().sum().values,
                           'Median log₂ abundance': log2.median().values}, index=log2.columns)
per_sample.index.name = 'Sample'
display(per_sample.style.format({'Proteins quantified': '{:,}', 'Median log₂ abundance': '{:.2f}'})
        .set_caption('Abundances by sample'))

correlation = log2.corr()
display(correlation.style.format('{:.3f}').background_gradient(cmap='Blues', axis=None, vmin=0.9, vmax=1)
        .set_caption('Correlation between the samples (log₂ abundances, Pearson)'))

### 4.3 Filtering and grouping: stimulated vs unstimulated
We keep the proteins quantified in at least two of the three replicates of both conditions, and compare the mean log₂ abundance of the stimulated (T) and unstimulated (U) samples: the log₂ fold change, with a Welch t-test and the Benjamini–Hochberg false discovery rate (FDR) over all the tested proteins.

In [ ]:
U, T = log2[[SAMPLE_NAMES[c] for c in UNSTIMULATED]], log2[[SAMPLE_NAMES[c] for c in STIMULATED]]
testable = (U.notna().sum(axis=1) >= 2) & (T.notna().sum(axis=1) >= 2)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _, p_values = stats.ttest_ind(T[testable], U[testable], axis=1, equal_var=False, nan_policy='omit')

def benjamini_hochberg(p):
    p = pd.Series(p)
    order = p.sort_values()
    adjusted = (order * len(p) / np.arange(1, len(p) + 1))[::-1].cummin()[::-1].clip(upper=1)
    return adjusted.reindex(p.index)

comparison = pd.DataFrame({
    'Gene': proteins.loc[testable, 'Gene'],
    'log₂ fold change (T/U)': T[testable].mean(axis=1) - U[testable].mean(axis=1),
    'p-value': np.asarray(p_values),
}, index=proteins.index[testable])
comparison['FDR'] = benjamini_hochberg(comparison['p-value']).values

tested = len(comparison)
nominal = (comparison['p-value'] < 0.05).sum()
display(pd.DataFrame({'Proteins': {
    'Tested': tested,
    'p < 0.05': nominal,
    'Expected p < 0.05 by chance alone': round(0.05 * tested),
    'p < 0.05 and fold change above 2 (|log₂ FC| > 1)': ((comparison['p-value'] < 0.05) & (comparison['log₂ fold change (T/U)'].abs() > 1)).sum(),
    'FDR < 0.05': (comparison['FDR'] < 0.05).sum(),
}}).style.format('{:,}').set_caption('Stimulated vs unstimulated'))

display(comparison.sort_values('p-value').head(10)
        .style.format({'log₂ fold change (T/U)': '{:+.2f}', 'p-value': '{:.4f}', 'FDR': '{:.2f}'})
        .set_caption('The ten smallest p-values (none significant after FDR correction)'))

With three replicates per condition, no protein changes significantly once the number of tests is accounted for, and fewer proteins reach p < 0.05 than chance alone would give: the vesicle proteome is very similar with and without stimulation, at the power of this design.

## 5. Visualization
- **Abundances by sample**, unstimulated in blue, stimulated in orange.
- **Correlation between the samples.**
- **Volcano plot** of the comparison: fold change against p-value, with the proteins at p < 0.05 and a fold change above 2 highlighted (none is significant after FDR correction).
- **Extracellular vesicle markers**: the abundance rank of canonical vesicle markers among all quantified proteins, a check of the vesicle preparation.

In [ ]:
CONDITION_COLORS = {'U': '#2a78d6', 'T': '#eb6834'}
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Abundances by sample
ax = axes[0, 0]
long = log2.melt(var_name='Sample', value_name='log₂ abundance').dropna()
sns.boxplot(data=long, x='Sample', y='log₂ abundance', hue='Sample', legend=False, fliersize=2, ax=ax,
            palette={name: CONDITION_COLORS[name.split()[1][0]] for name in log2.columns})
ax.set(title='Normalized protein abundances by sample', xlabel='')

# Correlation between the samples
ax = axes[0, 1]
sns.heatmap(correlation, annot=True, fmt='.3f', cmap='Blues', vmin=0.9, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson correlation (log₂)'})
ax.set_title('Correlation between the samples')

# Volcano plot
ax = axes[1, 0]
x, y = comparison['log₂ fold change (T/U)'], -np.log10(comparison['p-value'])
highlight = (comparison['p-value'] < 0.05) & (x.abs() > 1)
ax.scatter(x[~highlight], y[~highlight], s=8, color='#c3c2b7', alpha=0.6, label='Other proteins')
for direction, selected, color in [('Higher after stimulation', highlight & (x > 0), CONDITION_COLORS['T']),
                                   ('Lower after stimulation', highlight & (x < 0), CONDITION_COLORS['U'])]:
    ax.scatter(x[selected], y[selected], s=24, color=color, edgecolor='white', linewidth=0.5,
               label=f'{direction} (p < 0.05, FC > 2): {selected.sum()}')
for accession, row in comparison[highlight].nsmallest(8, 'p-value').iterrows():
    ax.annotate(row['Gene'], (row['log₂ fold change (T/U)'], -np.log10(row['p-value'])),
                xytext=(4, 3), textcoords='offset points', fontsize=8, color='#52514e')
ax.axhline(-np.log10(0.05), color='#898781', linestyle='--', linewidth=1)
for line in (-1, 1):
    ax.axvline(line, color='#898781', linestyle=':', linewidth=1)
ax.set(xlabel='log₂ fold change (stimulated / unstimulated)', ylabel='-log₁₀(p-value)',
       title=f'Stimulated vs unstimulated: {tested:,} proteins, none significant at FDR < 0.05')
ax.legend(fontsize=9, loc='upper right')

# Extracellular vesicle markers
ax = axes[1, 1]
MARKERS = {'CD63': 'tetraspanin', 'CD9': 'tetraspanin', 'CD81': 'tetraspanin', 'PDCD6IP': 'ALIX, ESCRT',
           'TSG101': 'ESCRT', 'SDCBP': 'syntenin-1', 'FLOT1': 'flotillin-1', 'HSPA8': 'HSC70', 'ANXA2': 'annexin A2'}
mean_abundance = log2.mean(axis=1)
rank = mean_abundance.rank(ascending=False)
markers = pd.DataFrame([{'Marker': f'{gene} ({role})', 'Rank': rank[proteins['Gene'] == gene].min(),
                         'Mean log₂ abundance': mean_abundance[proteins['Gene'] == gene].max()}
                        for gene, role in MARKERS.items() if (proteins['Gene'] == gene).any()]).sort_values('Rank')
bars = ax.barh(markers['Marker'][::-1], markers['Mean log₂ abundance'][::-1], color='#2a78d6', edgecolor='white', linewidth=2)
ax.bar_label(bars, labels=[f"rank {r:,.0f}" for r in markers['Rank'][::-1]], padding=3, fontsize=9)
ax.set(xlabel='Mean log₂ abundance', xlim=(mean_abundance.min(), mean_abundance.max() * 1.08),
       title=f'Canonical vesicle markers, ranked among {mean_abundance.notna().sum():,} proteins')

plt.tight_layout()
plt.show()

## 6. Conclusion
We loaded the mast cell extracellular vesicle proteomics dataset with the `mlcroissant` library, separated its nested protein and peptide levels, and explored the protein abundances:

- **Identification**: about 4,500 proteins, three quarters of them quantified in all six samples, with a median of six peptides each.
- **Reproducibility**: the replicates agree closely, and the stimulated samples correlate with the unstimulated ones as well as the replicates with each other.
- **Stimulation**: no protein changes significantly after correction for multiple testing; with three replicates per condition, only large and consistent changes could be detected.
- **Vesicle markers**: the canonical extracellular vesicle markers (tetraspanins CD63, CD9 and CD81, ALIX, TSG101, syntenin-1, flotillin-1) are all present, consistent with a vesicle preparation.

Further analyses can use the peptide level (modifications, quantification usage), impute the missing values to test more proteins, or compare the vesicle proteome with mast cell or vesicle reference proteomes.